In [11]:
# Imports
import requests
import pandas as pd
import time
import random
from bs4 import BeautifulSoup

In [12]:
# Seed keywords (English, target audience = Western EU/US) + locales
seed_keywords = [
    "lithium Bosnia",
    "Lopare lithium mine",
    "Bosnia mining",
    "Bosnia natural resources",
    "bauxite Bosnia",
    "coal mining Bosnia",
    "hydropower Bosnia",
    "Bosnia forests",
    "Bosnia water resources",
    "rare earth minerals Balkans",
    "Bosnia and Herzegovina minerals",
    "Bosnia mining investment",
    "lithium mining Europe",
    "Balkan natural resources",
    "Bosnia iron ore",
    "Zenica steel Bosnia",
    "Bosnia zinc mining",
    "Bosnia lead mining",
    "Tuzla salt mines",
    "Bosnia timber industry",
    "Bosnia wood export",
    "Bosnia wind energy",
    "Bosnia solar energy",
    "Bosnia natural gas",
    "Bosnia oil reserves",
    "Bosnia copper mining",
    "Bosnia manganese",
    "Bosnia agriculture land",
]

locales = [
    {"gl": "us", "hl": "en"},  # USA
    {"gl": "gb", "hl": "en"},  # UK
    {"gl": "de", "hl": "en"},  # Germany
    {"gl": "at", "hl": "en"},  # Austria
    {"gl": "fr", "hl": "en"},  # France
    {"gl": "it", "hl": "en"},  # Italy
    {"gl": "nl", "hl": "en"},  # Netherlands
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [13]:
# Google Autocomplete scraper
def get_autocomplete_suggestions(query, gl="us", hl="en"):
    url = "http://suggestqueries.google.com/complete/search"
    params = {"client": "firefox", "q": query, "gl": gl, "hl": hl}
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=5)
        r.raise_for_status()
        return r.json()[1]
    except Exception as e:
        print(f"[Autocomplete] Error for '{query}' ({gl}): {e}")
        return []

In [ ]:
# Related searches via SerpApi (with graceful fallback)
SERPAPI_KEY = ""  # paste your key here if you have one, otherwise leave empty

def get_related_searches(query, gl="us", hl="en"):
    if not SERPAPI_KEY or not SERPAPI_KEY.strip():
        return []

    url = "https://serpapi.com/search"
    params = {"q": query, "gl": gl, "hl": hl, "api_key": SERPAPI_KEY}

    for attempt in range(4):
        try:
            r = requests.get(url, params=params, timeout=45)

            if r.status_code == 401:
                print("[RelatedSearches] Invalid API key - skipping.")
                return []

            if r.status_code == 429:
                wait = 5 * (attempt + 1)
                print(f"[RelatedSearches] 429 za '{query}' ({gl}), cekam {wait}s (pokusaj {attempt + 1}/4)")
                time.sleep(wait)
                continue

            r.raise_for_status()
            data = r.json()

            if "error" in data:
                print(f"[RelatedSearches] API error: {data['error']} - skipping.")
                return []

            return [item["query"] for item in data.get("related_searches", [])]
        except Exception as e:
            print(f"[RelatedSearches] Error za '{query}' ({gl}): {e}")
            time.sleep(3)

    print(f"[RelatedSearches] Odustajem za '{query}' ({gl}) nakon 4 pokusaja")
    return []

In [15]:
#Zadnja koju pokrecem
remaining_locales = [
    {"gl": "gb", "hl": "en"},
    {"gl": "de", "hl": "en"},
    {"gl": "at", "hl": "en"},
    {"gl": "fr", "hl": "en"},
    {"gl": "it", "hl": "en"},
    {"gl": "nl", "hl": "en"},
]

rel_path = "keywords_related_remaining.csv"
rows_rel = []

for seed in seed_keywords:
    for loc in remaining_locales:
        for s in get_related_searches(seed, gl=loc["gl"], hl=loc["hl"]):
            rows_rel.append({
                "seed_keyword": seed,
                "suggested_keyword": s,
                "source": "related_search",
                "country_sim": loc["gl"],
            })
        time.sleep(random.uniform(1.5, 3.0))
    pd.DataFrame(rows_rel).to_csv(rel_path, index=False)
    print(f"{seed}: related redova = {len(rows_rel)}")

print(f"Gotovo. Related redova: {len(rows_rel)}")
if rows_rel:
    print(pd.DataFrame(rows_rel)["country_sim"].value_counts())

lithium Bosnia: related redova = 13
Lopare lithium mine: related redova = 23
Bosnia mining: related redova = 71
Bosnia natural resources: related redova = 107
[RelatedSearches] Error za 'bauxite Bosnia' (it): HTTPSConnectionPool(host='serpapi.com', port=443): Read timed out. (read timeout=45)
bauxite Bosnia: related redova = 109
coal mining Bosnia: related redova = 157
hydropower Bosnia: related redova = 187
Bosnia forests: related redova = 235
Bosnia water resources: related redova = 247
rare earth minerals Balkans: related redova = 247
Bosnia and Herzegovina minerals: related redova = 253
Bosnia mining investment: related redova = 268
lithium mining Europe: related redova = 316
Balkan natural resources: related redova = 364
Bosnia iron ore: related redova = 412
Zenica steel Bosnia: related redova = 460
Bosnia zinc mining: related redova = 504
Bosnia lead mining: related redova = 546
[RelatedSearches] Error za 'Tuzla salt mines' (gb): HTTPSConnectionPool(host='serpapi.com', port=443):

In [7]:
df_new = pd.DataFrame(rows_new)
print(f"Redova: {len(df_new)}")
print(f"Jedinstvenih keyworda: {df_new['suggested_keyword'].nunique()}")

Redova: 540
Jedinstvenih keyworda: 86


In [ ]:
# Run scraper across seeds and locales
rows = []

for seed in seed_keywords:
    for loc in locales:
        # Autocomplete
        for s in get_autocomplete_suggestions(seed, gl=loc["gl"], hl=loc["hl"]):
            rows.append({
                "seed_keyword": seed,
                "suggested_keyword": s,
                "source": "autocomplete",
                "country_sim": loc["gl"]
            })

        time.sleep(random.uniform(1.0, 2.0))

        # Related searches
        for s in get_related_searches(seed, gl=loc["gl"], hl=loc["hl"]):
            rows.append({
                "seed_keyword": seed,
                "suggested_keyword": s,
                "source": "related_search",
                "country_sim": loc["gl"]
            })

        time.sleep(random.uniform(1.5, 3.0))

print(f"Collected {len(rows)} rows")

In [ ]:
# Build DataFrame, clean, save
df = pd.DataFrame(rows)
df_unique = df.drop_duplicates(subset="suggested_keyword").reset_index(drop=True)

print(f"Total raw rows: {len(df)}")
print(f"Unique keywords: {len(df_unique)}")
print(df_unique["source"].value_counts())

output_path = "keywords_natural_resources_bih.csv"
df_unique.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

df_unique.head(20)

In [ ]:
# List all unique suggested keywords to identify off-topic/junk keywords for cleanup step
for kw in df_unique["suggested_keyword"]:
  print(kw)

In [ ]:
# Cleanup - remove off-topic/junk keywords
junk_keywords = [
    # Landmines / unrelated mining tangents
    "Bosnia landmines map 2025",
    "Bosnia landmines 2024",
    "Why are there so many landmines in Bosnia",
    "Bosnia mine map app",
    "Bosnia mining disaster",
    "Who put landmines in bosnia",
    "Bosnia landmine deaths per year",
    "Bosnia landmines map",
    "Where are landmines in bosnia",
    "Bosnia mining rig",
    "Bosnia copper mining rig",
    "bitcoin mining bosnia",
    "low lithium levels while taking lithium",

    # General country facts / politics / economy (not natural resources)
    "is bosnia a third world country",
    "is bosnia a poor country",
    "national food of bosnia",
    "Where is Bosnia currency",
    "Bosnia and herzegovina border countries",
    "Bosnia is located in which country",
    "Bosnia and Herzegovina",
    "is bosnia a democratic country",
    "is bosnia a democracy",
    "is bosnia a good country",
    "is bosnia a free country",
    "bosnia tax rate",
    "bosnia and herzegovina tax rate",
    "how wealthy is bosnia and herzegovina",
    "taxes in bosnia",
    "bosnia median income",
    "bosnia under which country",
    "is bosnia in the balkans",
    "Bosnia and Herzegovina travel guide",
    "Republic of Bosnia and Herzegovina",
    "Bosnija",
    "Bosnia and Herzegovina Map",
    "History of Bosnia",
    "Bosnia today",
    "Mostar wiki",
    "Bosnia religion",
    "Bosnia football",
    "Bosnia currency",
    "Bosnia map",
    "Bosnia and Herzegovina vs",
    "Bosnia people",
    "Bosnia in Europe",
    "Bosnia and herzegovina trade",
    "Us trade with bosnia",
    "Bosnian economy",

    # Balkans general / unrelated
    "what is balkan",
    "are balkan countries safe",
    "Balkan countries",
    "How many Balkan countries are there",
    "Why is it called the Balkans",
    "Balkan people",
    "What are the 11 Balkan countries",
    "Balkan natural resources albania",
    "List of balkan natural resources",
    "Balkan natural resources notes",

    # Rare earths - too generic/global, not B&H specific
    "Rare earth minerals balkans wikipedia",
    "Rare earth minerals balkans list",
    "Rare earth minerals sweden 2025",
    "Rare earth mining by country",
    "Australian reserves of rare earth minerals",
    "How to find rare earth minerals",
    "Rare earth in arctic",
    "Huge rare earth metals discovery in arctic sweden",
    "Yttrium deposits by country",
    "Is china the only country with rare earth minerals",
    "European rare earth companies",
    "Rare earths europe map",
    "Rare earth elements",
    "Rare earth magnets and motors a european call for action",
    "Critical raw materials Act",

    # Lithium - too generic/global, not B&H specific
    "what country has the most lithium",
    "biggest lithium countries",
    "which country has the most lithium mines",
    "Lithium production by country",
    "Lithium reserves by country",
    "European Lithium stock",
    "Largest lithium deposits in europe",
    "Europe lithium demand",
    "Portugal lithium reserves",
    "European Lithium chart",
    "European Lithium Ltd",
    "Who owns European Lithium",
    "European Lithium stock ASX",
    "Lithium mining europe pdf",
    "Is european lithium a good investment",
    "European Lithium announcements",
    "Lithium supply chains",
    "Lithium commodity",
    "Lithium market supply and demand",
    "Lithium demand and supply forecast",
    "Lithium mining portugal map",
    "coal mining by country",
    "which countries produce the most hydroelectric power",
    "hydropower capacity by country",

    # Forests - off topic
    "Bosnia forests weather",
    "Bosnia forests holidays",
    "Bosnia forests animals",
    "Primeval forests in Europe",
    "Is perucica a rainforest",
    "Bosnia water resources phone number",
    "Bosnia water resources wikipedia",

    # Zenica/steel - mostly off-topic geography
    "Zenica steel bosnia address",
    "Zenica bosnia and herzegovina",
    "Why does bosnia play in zenica",
    "Zenica bosnia currency",
    "Where is Zenica in which country",
    "Zenica population",
    "Zenica map",
    "Zenica Bosnia",
    "Zenica religion",
    "Zenica Bosnia demographics",
    "ArcelorMittal owner",

    # Tuzla salt - off-topic tangents
    "what are salt mines used for",
    "are salt mines safe",
    "what are salt mines",
    "Map of Sarajevo",
    "Municipalities of bosnia",
    "Hac city",
    "Sarajevo mountains",
    "Sarajevo summer",
    "Facts about mostar",
    "Tuzlanska kapija",
    "Tuzla salt mines tours",
    "Tuzla salt mines wikipedia",

    # Solar - off-topic
    "solar energy degree programs",
    "solar energy which country",
    "Future prospects of renewable energy",

    # Oil - too generic/global comparisons
    "does russia have oil reserves",
    "does brazil have oil reserves",
    "which country has the smallest oil reserves",
    "how long will russia's oil reserves last",
    "European oil fields",

    # Agriculture land - real estate listings, off-topic
    "Bosnia agriculture land for sale",
    "Bosnia agriculture land price",
    "Bosnia agriculture land prices",
    "Bosnia agriculture land nekretnine",
    "Land for sale in Bosnia",
    "Farm house for sale in bosnia",
    "Bosnia agriculture land jobs",
]

df_clean = df_unique[~df_unique["suggested_keyword"].isin(junk_keywords)].reset_index(drop=True)

print(f"Before cleanup: {len(df_unique)}")
print(f"After cleanup: {len(df_clean)}")

df_clean.to_csv("keywords_natural_resources_bih.csv", index=False)
print("Saved cleaned CSV")
df_clean

In [ ]:
# CELL 9: Resource category statistics
import pandas as pd
import re

# Load from saved CSV
df_clean = pd.read_csv("keywords_natural_resources_bih.csv")

# Define resource categories and their keywords
categories = {
    "Lithium": ["lithium"],
    "Coal": ["coal"],
    "Iron & Steel": ["iron", "steel"],
    "Bauxite": ["bauxite"],
    "Zinc & Lead": ["zinc", "lead"],
    "Copper": ["copper"],
    "Salt": ["salt"],
    "Timber & Wood": ["timber", "wood", "forest"],
    "Hydropower": ["hydropower", "hydro"],
    "Wind & Solar Energy": ["wind", "solar", "renewable"],
    "Natural Gas & Oil": ["gas", "oil"],
    "Rare Earth & Minerals": ["rare earth", "mineral", "manganese"],
    "Agriculture": ["agriculture"],
    "General Mining": ["mining", "mine"],
}

def assign_category(keyword):
    kw = keyword.lower()
    for category, terms in categories.items():
        if any(term in kw for term in terms):
            return category
    return "Other"

df_clean["category"] = df_clean["suggested_keyword"].apply(assign_category)

# Build stats table
stats = df_clean.groupby("category").agg(
    keyword_count=("suggested_keyword", "count"),
    us_count=("country_sim", lambda x: (x == "us").sum()),
    gb_count=("country_sim", lambda x: (x == "gb").sum()),
    de_count=("country_sim", lambda x: (x == "de").sum()),
    at_count=("country_sim", lambda x: (x == "at").sum()),
    fr_count=("country_sim", lambda x: (x == "fr").sum()),
    it_count=("country_sim", lambda x: (x == "it").sum()),
    nl_count=("country_sim", lambda x: (x == "nl").sum()),
    autocomplete_count=("source", lambda x: (x == "autocomplete").sum()),
    related_search_count=("source", lambda x: (x == "related_search").sum()),
).reset_index()

stats.columns = [
    "Category", "Total Keywords",
    "US", "UK", "DE", "AT", "FR", "IT", "NL",
    "Autocomplete", "Related Search"
]

stats = stats.sort_values("Total Keywords", ascending=False).reset_index(drop=True)

print(f"Total keywords: {len(df_clean)}")
print(f"Total categories: {len(stats)}\n")
stats

Total keywords: 164
Total categories: 15



,Category,Total Keywords,US,UK,DE,AT,FR,IT,NL,Autocomplete,Related Search
0,Other,35,24,8,1,1,0,1,0,7,28
1,General Mining,23,20,2,0,0,1,0,0,10,13
2,Lithium,22,21,0,0,0,1,0,0,16,6
3,Natural Gas & Oil,18,13,4,0,1,0,0,0,6,12
4,Timber & Wood,12,7,1,4,0,0,0,0,1,11
5,Coal,10,9,0,1,0,0,0,0,2,8
6,Wind & Solar Energy,10,7,2,0,1,0,0,0,5,5
7,Iron & Steel,7,4,2,0,0,1,0,0,1,6
8,Bauxite,6,1,3,0,0,0,0,2,1,5
9,Rare Earth & Minerals,5,5,0,0,0,0,0,0,2,3


In [24]:
#Merge both runs into one csv
old = pd.read_excel("keywords_natural_resources_bih.xlsx")
rel = pd.read_csv("keywords_related_remaining.csv", encoding="utf-8")


def fix_mojibake(s):
    if not isinstance(s, str):
        return s
    try:
        return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


junk_keywords = [
    "Bosnia landmines map 2025", "Bosnia landmines 2024", "Why are there so many landmines in Bosnia",
    "Bosnia mine map app", "Bosnia mining disaster", "Who put landmines in bosnia",
    "Bosnia landmine deaths per year", "Bosnia landmines map", "Where are landmines in bosnia",
    "Bosnia mining rig", "Bosnia copper mining rig", "bitcoin mining bosnia", "low lithium levels while taking lithium",
    "is bosnia a third world country", "is bosnia a poor country", "national food of bosnia",
    "Where is Bosnia currency", "Bosnia and herzegovina border countries", "Bosnia is located in which country",
    "Bosnia and Herzegovina", "is bosnia a democratic country", "is bosnia a democracy", "is bosnia a good country",
    "is bosnia a free country", "bosnia tax rate", "bosnia and herzegovina tax rate", "how wealthy is bosnia and herzegovina",
    "taxes in bosnia", "bosnia median income", "bosnia under which country", "is bosnia in the balkans",
    "Bosnia and Herzegovina travel guide", "Republic of Bosnia and Herzegovina", "Bosnija", "Bosnia and Herzegovina Map",
    "History of Bosnia", "Bosnia today", "Mostar wiki", "Bosnia religion", "Bosnia football", "Bosnia currency", "Bosnia map",
    "Bosnia and Herzegovina vs", "Bosnia people", "Bosnia in Europe", "Bosnia and herzegovina trade", "Us trade with bosnia",
    "Bosnian economy", "what is balkan", "are balkan countries safe", "Balkan countries", "How many Balkan countries are there",
    "Why is it called the Balkans", "Balkan people", "What are the 11 Balkan countries", "Balkan natural resources albania",
    "List of balkan natural resources", "Balkan natural resources notes", "Rare earth minerals balkans wikipedia",
    "Rare earth minerals balkans list", "Rare earth minerals sweden 2025", "Rare earth mining by country",
    "Australian reserves of rare earth minerals", "How to find rare earth minerals", "Rare earth in arctic",
    "Huge rare earth metals discovery in arctic sweden", "Yttrium deposits by country",
    "Is china the only country with rare earth minerals", "European rare earth companies", "Rare earths europe map",
    "Rare earth elements", "Rare earth magnets and motors a european call for action", "Critical raw materials Act",
    "what country has the most lithium", "biggest lithium countries", "which country has the most lithium mines",
    "Lithium production by country", "Lithium reserves by country", "European Lithium stock",
    "Largest lithium deposits in europe", "Europe lithium demand", "Portugal lithium reserves", "European Lithium chart",
    "European Lithium Ltd", "Who owns European Lithium", "European Lithium stock ASX", "Lithium mining europe pdf",
    "Is european lithium a good investment", "European Lithium announcements", "Lithium supply chains", "Lithium commodity",
    "Lithium market supply and demand", "Lithium demand and supply forecast", "Lithium mining portugal map",
    "coal mining by country", "which countries produce the most hydroelectric power", "hydropower capacity by country",
    "Bosnia forests weather", "Bosnia forests holidays", "Bosnia forests animals", "Primeval forests in Europe",
    "Is perucica a rainforest", "Bosnia water resources phone number", "Bosnia water resources wikipedia",
    "Zenica steel bosnia address", "Zenica bosnia and herzegovina", "Why does bosnia play in zenica", "Zenica bosnia currency",
    "Where is Zenica in which country", "Zenica population", "Zenica map", "Zenica Bosnia", "Zenica religion",
    "Zenica Bosnia demographics", "ArcelorMittal owner", "what are salt mines used for", "are salt mines safe", "what are salt mines",
    "Map of Sarajevo", "Municipalities of bosnia", "Hac city", "Sarajevo mountains", "Sarajevo summer", "Facts about mostar",
    "Tuzlanska kapija", "Tuzla salt mines tours", "Tuzla salt mines wikipedia", "solar energy degree programs",
    "solar energy which country", "Future prospects of renewable energy", "does russia have oil reserves",
    "does brazil have oil reserves", "which country has the smallest oil reserves", "how long will russia's oil reserves last",
    "European oil fields", "Bosnia agriculture land for sale", "Bosnia agriculture land price", "Bosnia agriculture land prices",
    "Bosnia agriculture land nekretnine", "Land for sale in Bosnia", "Farm house for sale in bosnia", "Bosnia agriculture land jobs",
    "Bosnia girls", "Balkan Sea", "Balkan Peninsula", "Zenitsu bosnia", "Is Sarajevo Muslim",
    "Bosnia landmines meme", "Bosnia mines meme", "Tuzla to Sarajevo", "What happened in Mostar",
    "What happened in mostar", "Zenica stadium", "Bosnia language", "Bosnia coastline map",
    "Bosnia Qatar", "How many landmines in Bosnia", "Who put landmines in Bosnia", "HAC city",
    "Is tuzla safe", "Tuzla currency", "Tuzla language", "Tuzla is in which country", "Mostar historia",
    "Zenica currency", "Zenica grad", "Zenica meaning", "Zenica bosnia weather", "Zenica, Bosnia",
    "Tuzla, Bosnia", "Sanski most mine", "Gacko Bosnia", "Bialowieza Forest", "Białowieża Forest",
    "Lipiczanska forest", "Croatia hydropower", "Montenegro power plants", "Chelopech mine",
    "Chelopech mine life", "Dundee Precious Metals", "DPM Metals", "Dpm bosnia", "Bosnia lead mining rig",
    "Bosnia zinc mining rig", "Power plants in bosnia",
]

df_combined = pd.concat([old, rel], ignore_index=True)
df_combined["suggested_keyword"] = df_combined["suggested_keyword"].apply(fix_mojibake)
df_combined = df_combined[~df_combined["suggested_keyword"].isin(junk_keywords)]
df_combined = df_combined.drop_duplicates(
    subset=["suggested_keyword", "country_sim"]
).reset_index(drop=True)

df_combined.to_excel("keywords_natural_resources_bih_combined_clean.xlsx", index=False)

print(f"Redova: {len(df_combined)}")
print(f"Jedinstvenih keyworda: {df_combined['suggested_keyword'].nunique()}")
print(df_combined["country_sim"].value_counts())

Redova: 589
Jedinstvenih keyworda: 194
country_sim
us    116
gb     91
nl     84
de     82
at     75
it     75
fr     66
Name: count, dtype: int64
